# Оценка фокуса составной рентгеновской линзы CRL

Задайте параметры в формах и выберите **Среда выполнения → Выполнить все**.
Установка окружения выполняется первой ячейкой. Для первого запуска подходят CPU и стандартные параметры.
При желании выберите GPU в настройках среды выполнения перед запуском.

Вы получите положение максимума интенсивности, FWHM, пропускание, графики и архив результатов.
Первый пример — бериллий, 12 кэВ, 10 элементов, источник в 15 м от входа CRL.

**Геометрия имеет физический смысл:** 1D описывает цилиндрическую линзу с фокусировкой по одной оси;
2D — осесимметричную линзу с круглой апертурой. Это разные модели. «Быстро / Точнее» управляет сеткой.

Расстояние от источника измеряется до входа CRL; расстояния до детектора — от выхода.
Входной пучок ограничен номинальной апертурой. Интенсивность относительная, амплитуда на входе равна единице.
Модель скалярная, монохроматическая и параксиальная; источник точечный либо плоская волна.

In [ ]:
#@title 1. Подготовить XPOSim — запустите один раз в новой сессии
import hashlib
import importlib.util
import json
import re
import subprocess
import sys
import urllib.request
from pathlib import Path
from IPython.display import display, Markdown

# Проверенная неизменяемая версия кода. Ноутбук не загружает изменяемый main.
SOURCE_REF = '9ffebc47f9515fd2f6e2a6ce39ac3a260e6d1a77'
SOURCE_SHA256 = 'db617a4c754056571f9d2356c9d32dde1b3f1634f4fe58014e9f3588f01cf23e'
PROJECT_DIR = Path.cwd() / '.xposim_runtime' / SOURCE_REF
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in ('numpy', 'scipy', 'matplotlib')
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--no-cache-dir', *missing])

# Colab обычно уже содержит CuPy. Установка нужна только при видимом GPU
# и отсутствии пакета; драйвер и системный CUDA Toolkit не заменяются.
try:
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=15)
except (FileNotFoundError, subprocess.TimeoutExpired):
    gpu_info = None
if gpu_info is not None and gpu_info.returncode == 0 and importlib.util.find_spec('cupy') is None:
    try:
        toolkit = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, timeout=15)
        release = re.search(r'release\s+(\d+)\.', toolkit.stdout)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        release = None
    if release is None:
        release = re.search(r'CUDA Version:\s*(\d+)\.', gpu_info.stdout)
    major = int(release.group(1)) if release else None
    wheel = {12: 'cupy-cuda12x', 13: 'cupy-cuda13x'}.get(major)
    if wheel:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                                   '--no-cache-dir', f'{wheel}==14.2.0'])
            importlib.invalidate_caches()
        except subprocess.CalledProcessError:
            print('Установка CuPy не удалась. Автоматический режим продолжит работу на CPU.')
    else:
        print('Версия CUDA не распознана. Будет использован CPU, если CuPy недоступен.')

files_to_fetch = [
    'XPOSim.py', 'xray_material_data/AtomicConstants.dat',
    'xray_material_data/CrossSec-Compton_McMaster.dat',
    'xray_material_data/f1f2_Windt.dat',
]
for relative in files_to_fetch:
    target = PROJECT_DIR / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        url = f'https://raw.githubusercontent.com/mamichberdey/XPOSim/{SOURCE_REF}/{relative}'
        with urllib.request.urlopen(url, timeout=90) as response:
            content = response.read()
        temporary = target.with_suffix(target.suffix + '.download')
        temporary.write_bytes(content)
        temporary.replace(target)
actual_hash = hashlib.sha256((PROJECT_DIR / 'XPOSim.py').read_bytes()).hexdigest()
if actual_hash != SOURCE_SHA256:
    raise RuntimeError('Контрольная сумма XPOSim не совпала. Удалите .xposim_runtime и повторите подготовку.')

spec = importlib.util.spec_from_file_location('XPOSim', PROJECT_DIR / 'XPOSim.py')
xpos = importlib.util.module_from_spec(spec)
sys.modules['XPOSim'] = xpos
spec.loader.exec_module(xpos)
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

available_backend = xpos.select_backend('auto')
print(f'XPOSim {xpos.__version__}; доступен {"GPU / CuPy" if available_backend == "cupy" else "CPU / NumPy"}.')
print(f'Версия исходников: {SOURCE_REF}')
if available_backend == 'numpy':
    print('CPU подходит для первого расчёта. GPU можно выбрать в настройках среды выполнения Colab.')
display(Markdown('**Подготовка завершена.** Задайте параметры в следующей форме.'))


## Параметры расчёта

Откройте форму следующей ячейки. Длины линзы вводятся в **мкм**, энергия в **кэВ**, расстояние до источника в **м**.
Меняя стандартный материал, вы одновременно выбираете его плотность. Для своей формулы задайте плотность отдельно.

Сначала выполните пример без изменений. Затем меняйте по одному параметру и повторно запускайте ячейки 2–4.
Слишком большое число элементов или источник внутри фокусного расстояния могут не дать реального внешнего изображения.

In [ ]:
#@title 2. Параметры источника и CRL
#@markdown ### Физическая модель
geometry = 'Цилиндрическая линза — 1D' #@param ["Цилиндрическая линза — 1D", "Осесимметричная линза — 2D"]
source = 'Точечный источник' #@param ["Точечный источник", "Плоская волна"]
energy_kev = 12.0 #@param {"type": "number"}
source_distance_m = 15.0 #@param {"type": "number"}
#@markdown Энергия — **кэВ**; расстояние от источника до входа CRL — **м**. Для плоской волны расстояние не используется.
#@markdown ### Материал
material_preset = 'Be — бериллий' #@param ["Be — бериллий", "Si — кремний", "Al — алюминий", "C — алмаз", "Свой материал"]
custom_formula = 'SiO2' #@param {"type": "string"}
custom_density_g_cm3 = 2.2 #@param {"type": "number"}
#@markdown Формула и плотность **г/см³** используются только при выборе «Свой материал».
#@markdown ### Геометрия — все длины ниже в мкм
radius_um = 50.0 #@param {"type": "number"}
aperture_um = 100.0 #@param {"type": "number"}
bridge_thickness_um = 30.0 #@param {"type": "number"}
n_elements = 10 #@param {"type": "integer"}
#@markdown `radius` — радиус кривизны; `aperture` — полный диаметр/ширина; `bridge_thickness` — минимальная толщина между поверхностями одного элемента.
#@markdown ### Расчёт
accuracy = 'Быстро' #@param ["Быстро", "Точнее"]
backend = 'Автоматически' #@param ["Автоматически", "CPU", "GPU"]
scan_mode = 'Автоматически' #@param ["Автоматически", "Свой диапазон"]
scan_start_mm = 500.0 #@param {"type": "number"}
scan_stop_mm = 1500.0 #@param {"type": "number"}
scan_points = 41 #@param {"type": "integer"}
#@markdown Расстояния скана — **мм от выхода CRL**; используются только для своего диапазона. Внутри интервала выполняется дополнительное уточнение максимума.
#@markdown ### Дополнительные настройки
#@markdown Нули означают автоматический выбор. «Точнее» увеличивает сетку, но не меняет физическую модель.
grid_points = 0 #@param {"type": "integer"}
window_factor = 2.0 #@param {"type": "number"}
batch_size = 0 #@param {"type": "integer"}

presets = {'Be — бериллий': ('Be', 1.848), 'Si — кремний': ('Si', 2.33),
           'Al — алюминий': ('Al', 2.70), 'C — алмаз': ('C', 3.515)}
formula, density = presets.get(material_preset, (custom_formula, custom_density_g_cm3))
if grid_points < 0 or batch_size < 0:
    raise ValueError('Число точек и размер батча должны быть положительными либо равны нулю для автонастройки.')
config = xpos.CRLFocusConfig(
    energy_kev=energy_kev, material_formula=formula, density_g_cm3=density,
    radius_m=radius_um * 1e-6, aperture_m=aperture_um * 1e-6,
    bridge_thickness_m=bridge_thickness_um * 1e-6, n_elements=n_elements,
    source_distance_m=source_distance_m if source == 'Точечный источник' else None,
    geometry='cylindrical' if geometry == 'Цилиндрическая линза — 1D' else 'axisymmetric',
    accuracy='preview' if accuracy == 'Быстро' else 'refined',
    backend={'Автоматически': 'auto', 'CPU': 'numpy', 'GPU': 'cupy'}[backend],
    scan_range_m=None if scan_mode == 'Автоматически' else (scan_start_mm * 1e-3, scan_stop_mm * 1e-3),
    scan_points=scan_points, grid_points=grid_points or None,
    window_factor=window_factor, batch_size=batch_size or None,
)
print(f'Параметры готовы: {formula}, {energy_kev:g} кэВ, {n_elements} элементов, {config.geometry}.')


In [ ]:
#@title Необязательно: загрузить ранее сохранённый configuration.json
restore_parameters = False #@param {"type": "boolean"}
#@markdown Включите флажок и запустите ячейку. Загруженный файл заменит параметры формы для следующего расчёта.
if restore_parameters:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Загрузка через эту форму доступна в Google Colab.') from None
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Выберите один configuration.json или metadata.json.')
    payload = json.loads(next(iter(uploaded.values())).decode('utf-8'))
    config = xpos.CRLFocusConfig(**payload.get('configuration', payload))
    print('Параметры загружены. Запустите расчёт ниже.')


## Результаты

Фокус определяется по **максимуму пиковой интенсивности на выбранных плоскостях**, с локальным уточнением.
Он может отличаться от минимума FWHM и аналитического геометрического фокуса.
Автоматический скан охватывает 0,5–1,5 аналитического расстояния. Если пик попал на границу, расширьте интервал.

«Пропускание» — отношение выходной мощности к мощности после входной апертуры, в конечном расчётном окне.

In [ ]:
#@title 3. Рассчитать фокус и показать результаты
import html
import time
from matplotlib.colors import LogNorm
from IPython.display import HTML

result = None
started = time.perf_counter()
try:
    result = xpos.estimate_crl_focus(config)
except (ValueError, RuntimeError, MemoryError, ArithmeticError) as exc:
    display(HTML('<p style="color:#b91c1c"><b>Расчёт остановлен.</b> Проверьте параметры. '
                 'При нехватке памяти уменьшите батч или сетку. Подробности: '
                 + html.escape(str(exc)) + '</p>'))
    raise
elapsed_s = time.perf_counter() - started
widths = np.atleast_1d(result.fwhm_at_peak_m)
width_text = ' × '.join('не определена' if w is None else f'{w * 1e6:.4g}' for w in widths)
rows = [
    ('Материал / энергия / элементов', f'{result.configuration.material_formula} / {result.configuration.energy_kev:g} кэВ / {result.configuration.n_elements}'),
    ('Модель', result.configuration.geometry),
    ('Найденный максимум, мм от выхода CRL', f'{result.peak_z_m * 1e3:.5g}'),
    ('Аналитическая оценка, мм от выхода CRL', f'{result.analytical_focus_m * 1e3:.5g}'),
    ('FWHM в найденной плоскости, мкм', width_text),
    ('Пропускание после входной апертуры', f'{result.transmission:.2%}'),
    ('Пиковая интенсивность, отн. ед.', f'{result.peak_intensity[result.peak_index]:.5g}'),
    ('Backend / сетка', f'{result.backend}; {len(result.x_m)}' + (' × ' + str(len(result.y_m)) if result.y_m is not None else '')),
    ('Шаг сетки, нм / время, с', f'{result.grid_step_m * 1e9:.4g} / {elapsed_s:.2f}'),
]
display(HTML('<table>' + ''.join('<tr><td>' + html.escape(a) + '</td><td><b>' + html.escape(b) + '</b></td></tr>' for a, b in rows) + '</table>'))
messages = {
    'grid_cap': 'Автоматическая сетка достигла ограничения памяти. Увеличьте число точек вручную и сравните результат.',
    'scan_boundary': 'Максимум находится на границе скана. Расширьте диапазон: положение фокуса пока не локализовано.',
    'fwhm_missing': 'Не удалось найти обе границы FWHM. Проверьте окно и профиль пучка.',
    'sampling': 'На FWHM приходится менее шести точек. Увеличьте число точек сетки.',
    'phase_sampling': 'Шаг сетки слишком велик для изменения фазы. Увеличьте число точек.',
    'window_boundary': 'Излучение достигает границы окна. Увеличьте window_factor с сохранением шага сетки.',
    'low_transmission': 'Пропускание очень мало. Проверьте материал и суммарную толщину линзы.',
}
if result.diagnostics:
    display(HTML('<div style="color:#b45309"><b>Проверьте численные условия:</b><ul>' + ''.join('<li>' + html.escape(messages.get(m.split(':', 1)[0], m)) + '</li>' for m in result.diagnostics) + '</ul></div>'))
else:
    print('Автоматические проверки не выявили проблем. Для оценки точности сравните с более мелкой сеткой и большим окном.')

finite_widths = [float(w) for w in widths if w is not None and np.isfinite(w)]
half_view = min(len(result.x_m) * result.grid_step_m / 2,
                max(5 * max(finite_widths, default=0), 10 * result.grid_step_m))
x_mask = abs(result.x_m) <= half_view
z_mm = result.z_m * 1e3
normalization = max(float(result.peak_intensity[result.peak_index]), np.finfo(float).tiny)
figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
ax = axes[0, 0]
ax.plot(z_mm, result.peak_intensity / normalization)
ax.axvline(result.analytical_focus_m * 1e3, color='gray', ls='--', label='Аналитическая оценка')
ax.axvline(result.peak_z_m * 1e3, color='crimson', ls=':', label='Найденный максимум')
ax.set(xlabel='Расстояние от выхода CRL, мм', ylabel='Пиковая интенсивность / максимум', title='Поиск фокуса')
ax.legend()
ax.grid(alpha=.25)
ax = axes[0, 1]
mesh = ax.pcolormesh(result.x_m[x_mask] * 1e6, z_mm,
                     result.longitudinal_intensity[:, x_mask] / normalization,
                     shading='auto', norm=LogNorm(vmin=1e-4, vmax=1))
ax.set(xlabel='x, мкм', ylabel='Расстояние от выхода CRL, мм', title='Продольный профиль через центр')
figure.colorbar(mesh, ax=ax, label='I / Imax')
ax = axes[1, 0]
fwhm = result.fwhm_m
if fwhm.ndim == 1:
    ax.plot(z_mm, fwhm * 1e6, label='FWHM x')
else:
    ax.plot(z_mm, fwhm[:, 0] * 1e6, label='FWHM x')
    ax.plot(z_mm, fwhm[:, 1] * 1e6, '--', label='FWHM y')
ax.axvline(result.peak_z_m * 1e3, color='crimson', ls=':')
ax.set(xlabel='Расстояние от выхода CRL, мм', ylabel='FWHM, мкм', title='Размер пучка вдоль оси')
ax.legend()
ax.grid(alpha=.25)
ax = axes[1, 1]
if result.y_m is None:
    ax.plot(result.x_m[x_mask] * 1e6, result.best_intensity[x_mask] / normalization)
    ax.set(xlabel='x, мкм', ylabel='I / Imax', title='Профиль в найденной плоскости')
    ax.grid(alpha=.25)
else:
    y_mask = abs(result.y_m) <= half_view
    mesh = ax.pcolormesh(result.x_m[x_mask] * 1e6, result.y_m[y_mask] * 1e6,
                         result.best_intensity[np.ix_(y_mask, x_mask)] / normalization,
                         shading='auto', norm=LogNorm(vmin=1e-4, vmax=1))
    ax.set(xlabel='x, мкм', ylabel='y, мкм', title='Пятно в найденной плоскости')
    ax.set_aspect('equal')
    figure.colorbar(mesh, ax=ax, label='I / Imax')
plt.show()


In [ ]:
#@title 4. Сохранить результаты и при необходимости скачать ZIP
#@markdown Архив создаётся всегда. Для загрузки на компьютер включите флажок и запустите эту ячейку.
download_zip = False #@param {"type": "boolean"}
import csv
import platform
import zipfile
from dataclasses import asdict
from datetime import datetime, timezone
from IPython.display import FileLink

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
results_dir = Path.cwd() / 'xposim_results' / stamp
results_dir.mkdir(parents=True, exist_ok=True)
metadata = result.summary()
metadata.update({
    'source_ref': SOURCE_REF, 'source_sha256': SOURCE_SHA256,
    'created_at_utc': datetime.now(timezone.utc).isoformat(), 'elapsed_s': elapsed_s,
    'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                    'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__,
                    'cupy': xpos.cp.__version__ if xpos.CUPY_AVAILABLE else None},
})
(results_dir / 'configuration.json').write_text(json.dumps(asdict(result.configuration), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
(results_dir / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
with (results_dir / 'focus_scan.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['z_m', 'z_mm', 'peak_relative_intensity', 'fwhm_x_m', 'fwhm_y_m'])
    for z, peak, widths in zip(result.z_m, result.peak_intensity, result.fwhm_m):
        widths = np.atleast_1d(widths)
        wx = float(widths[0]) if np.isfinite(widths[0]) else ''
        wy = float(widths[1]) if len(widths) == 2 and np.isfinite(widths[1]) else ''
        writer.writerow([float(z), float(z * 1e3), float(peak), wx, wy])
np.savez_compressed(results_dir / 'arrays.npz', z_m=result.z_m, x_m=result.x_m,
                    y_m=np.array([]) if result.y_m is None else result.y_m,
                    peak_intensity=result.peak_intensity, fwhm_m=result.fwhm_m,
                    best_intensity=result.best_intensity,
                    longitudinal_intensity=result.longitudinal_intensity)
figure.savefig(results_dir / 'focus_report.png', dpi=180, bbox_inches='tight')
archive_path = results_dir.with_suffix('.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(results_dir.iterdir()):
        archive.write(path, arcname=path.name)
print('Архив результатов:', archive_path)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
if download_zip and colab_files is not None:
    colab_files.download(str(archive_path))
elif colab_files is None:
    display(FileLink(str(archive_path.relative_to(Path.cwd()))))
else:
    print('Архив доступен слева в разделе «Файлы» → xposim_results. Для скачивания из ячейки включите download_zip.')


## Как проверить результат и повторить расчёт

- Сравните режимы **Быстро** и **Точнее**. Само отсутствие предупреждений не подтверждает сходимость.
- Отдельно увеличьте физическое окно. При удвоении `window_factor` удвойте `grid_points`, чтобы сохранить шаг.
- Увеличьте число плоскостей скана и проверьте устойчивость положения максимума.
- Сообщение о границе скана означает, что фокус ещё не локализован. Сообщение о сетке требует её уточнения.
- На GPU размер батча выбирается автоматически; при нехватке памяти задайте `batch_size = 1`.

В ZIP находятся `configuration.json`, `metadata.json`, `focus_scan.csv`, `arrays.npz` и `focus_report.png`.
`arrays.npz` хранит численные массивы; `metadata.json` содержит версию исходников, зависимости и диагностику.
Архив сохраняется в `xposim_results`. Файлы сессии Colab временные: скачайте архив перед завершением работы.

Чтобы повторить расчёт, загрузите `configuration.json` в необязательной ячейке после формы и запустите ячейки 3–4.
После восстановления значения формы визуально не меняются: фактический материал и модель показаны в таблице результата.
Для возврата к форме отключите восстановление и вновь выполните ячейку 2.

Для запуска используется закреплённый commit XPOSim, указанный первой ячейкой. Повторный запуск не загружает изменяемый `main`.
Гайд для автора и пользователей: [COLAB_GUIDE_RU.md](https://github.com/mamichberdey/XPOSim/blob/codex/xposim-correctness-gpu-batching/COLAB_GUIDE_RU.md).